In [1]:
!pip install torch transformers datasets accelerate peft bitsandbytes
!pip install wandb tensorboard seqeval scikit-learn
!pip install sentencepiece protobuf
!!pip install jsonlines
!pip install -U unsloth trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.5 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=5d08def9d66603fe24a98f5fe7be4511a738ee76a60d04734c7ef9af3c7ab3cc
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.7/405.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import yaml
import torch
from transformers import (
    TrainingArguments,
    Trainer,
)
from torch.utils.data import DataLoader
from datasets import load_dataset, load_from_disk
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

2026-02-07 17:21:44.158096: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770484904.384766      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770484904.443668      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770484904.996346      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770484904.996387      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770484904.996390      55 computation_placer.cc:177] computation placer alr

In [3]:
DATASETS_DIR = "./datasets"
OUTPUTS_DIR = "./outputs"

os.makedirs(DATASETS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

In [4]:
from datasets import load_dataset
import json
import random
from pathlib import Path

# -----------------------------
# Config
# -----------------------------
DATASET_NAME = "ai4privacy/pii-masking-200k"
OUTPUT_DIR = Path("./pii_dataset_as_is")
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
TRAIN_RATIO = 0.7
DEV_RATIO = 0.2
TEST_RATIO = 0.1

random.seed(SEED)

# -----------------------------
# Load Dataset (ONLY train exists)
# -----------------------------
dataset = load_dataset(DATASET_NAME, split="train")

# -----------------------------
# Process examples (labels unchanged)
# -----------------------------
processed = []

for item in dataset:
    entities = []

    for span in item["privacy_mask"]:
        entities.append({
            "label": span["label"],  # unchanged
            "text": item["source_text"][span["start"]:span["end"]],
            # Uncomment if needed
            # "start": span["start"],
            # "end": span["end"],
        })

    if entities:
        processed.append({
            "text": item["source_text"],
            "entities": entities
        })

# -----------------------------
# Shuffle & Split
# -----------------------------
random.shuffle(processed)

n = len(processed)
train_end = int(n * TRAIN_RATIO)
dev_end = train_end + int(n * DEV_RATIO)

train_data = processed[:train_end]
dev_data   = processed[train_end:dev_end]
test_data  = processed[dev_end:]

# -----------------------------
# Save JSONL
# -----------------------------
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

save_jsonl(train_data, OUTPUT_DIR / "train.jsonl")
save_jsonl(dev_data, OUTPUT_DIR / "dev.jsonl")
save_jsonl(test_data, OUTPUT_DIR / "test.jsonl")

# -----------------------------
# Stats
# -----------------------------
print("✅ Done!")
print(f"Train: {len(train_data)}")
print(f"Dev:   {len(dev_data)}")
print(f"Test:  {len(test_data)}")


README.md: 0.00B [00:00, ?B/s]

english_pii_43k.jsonl:   0%|          | 0.00/73.8M [00:00<?, ?B/s]

french_pii_62k.jsonl:   0%|          | 0.00/116M [00:00<?, ?B/s]

german_pii_52k.jsonl:   0%|          | 0.00/97.8M [00:00<?, ?B/s]

italian_pii_50k.jsonl:   0%|          | 0.00/93.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/209261 [00:00<?, ? examples/s]

✅ Done!
Train: 146482
Dev:   41852
Test:  20927


In [18]:
dataset = load_dataset(
    "json",
    data_files={
        "train": os.path.join(OUTPUT_DIR,"train.jsonl"),
        "validation": os.path.join(OUTPUT_DIR,"dev.jsonl")
    }
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [5]:
def setup_model_and_tokenizer():
    """
    Load model, tokenizer, and apply quantization + LoRA config if specified.

    Args:
        use_4bit (bool, optional): Override whether to load in 4-bit mode.
        use_lora (bool, optional): Override whether to apply LoRA adapters.

    Returns:
        tuple: (model, tokenizer)
    """
    model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
    print(f"\nLoading model: {model_name}")

    # ------------------------------
    # Tokenizer setup
    # ------------------------------
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Determine quantization + LoRA usage
    load_in_4bit = True
    apply_lora = 'lora_r'

    # ------------------------------
    # Quantization setup (optional)
    # ------------------------------

    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )


    # ------------------------------
    # Model loading
    # ------------------------------
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_cfg,
        device_map="auto", #### device_map = "balanced",
        dtype=torch.bfloat16

    )

    # ------------------------------
    # LoRA setup (optional)
    # ------------------------------
    
    print("🔧 Applying LoRA configuration...")
    model = prepare_model_for_kbit_training(model)
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    # model.print_trainable_parameters()


    return model, tokenizer


In [6]:
model, tokenizer = setup_model_and_tokenizer()


Loading model: Qwen/Qwen2.5-1.5B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

🔧 Applying LoRA configuration...


In [13]:
task_instruction = """

You are a precise information extraction system.

Your task is to extract personally identifiable information (PII) entities from the given text.

Extract all the entities that you find in the text

Rules:
1. Return ONLY valid JSON. Do NOT include explanations or extra text.
5. The "text" field MUST match the substring from start to end.
6. If no entities are found, return: {"entities": []}
7. Do NOT guess or infer missing information.

Output format:
{
  "entities": [
    {
      "entity_type": "<one of the allowed types>",
      "text": "<exact substring>"
    }
  ]
}

Text:
{text}

"""

In [23]:
from datasets import load_dataset

def build_messages_for_sample(text, entities, task_instruction, include_assistant):
    
    messages = [
        {"role": "user", "content": task_instruction + " " + text},
        
    ]
    if include_assistant:
        messages.append({
            "role": "assistant",
            "content": json.dumps({"entities": entities}, ensure_ascii=False)
        })

    return messages

In [24]:
def preprocess_samples(examples, tokenizer, task_instruction, max_length):
    """Tokenise dialogues and apply assistant-only masking for causal LM."""
    input_ids_list, labels_list, attn_masks = [], [], []

    for text, entities in zip(examples["text"], examples["entities"]):
        sample = {"text": text, "entities": entities}

        # Build chat-style text

        msgs_full = build_messages_for_sample(
            text, entities, task_instruction, include_assistant=True
        )
        msgs_prompt = build_messages_for_sample(
            text, entities, task_instruction, include_assistant=False
        )

        text_full = tokenizer.apply_chat_template(
            msgs_full, tokenize=False, add_generation_prompt=False
        )
        text_prompt = tokenizer.apply_chat_template(
            msgs_prompt, tokenize=False, add_generation_prompt=True
        )
        prompt_len = len(text_prompt)

        tokens = tokenizer(
            text_full,
            max_length=max_length,
            truncation=True,
            padding=False,
            add_special_tokens=False,
            return_offsets_mapping=True,
        )

        # Mask non-assistant tokens
        start_idx = len(tokens["input_ids"])
        for i, (start, _) in enumerate(tokens["offset_mapping"]):
            if start >= prompt_len:
                start_idx = i
                break

        labels = [-100] * start_idx + tokens["input_ids"][start_idx:]
        input_ids_list.append(tokens["input_ids"])
        labels_list.append(labels)
        attn_masks.append(tokens["attention_mask"])

    return {
        "input_ids": input_ids_list,
        "labels": labels_list,
        "attention_mask": attn_masks,
    }

In [26]:
from torch.nn.utils.rnn import pad_sequence

class PaddingCollator:
    def __init__(self, tokenizer, label_pad_token_id=-100):
        self.tokenizer = tokenizer
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, batch):
        # Convert lists to tensors
        input_ids = [torch.tensor(f["input_ids"], dtype=torch.long) for f in batch]
        attn_masks = [torch.tensor(f["attention_mask"], dtype=torch.long) for f in batch]
        labels = [torch.tensor(f["labels"], dtype=torch.long) for f in batch]

        # Pad to the max length in this batch
        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        attn_masks = pad_sequence(attn_masks, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=self.label_pad_token_id)

        return {
            "input_ids": input_ids,
            "attention_mask": attn_masks,
            "labels": labels,
        }

In [27]:
def tokenize_dataset(tokenizer, train_data, val_data):

    print("\nTokenizing datasets...")
    tokenized_train = train_data.map(
        lambda e: preprocess_samples(
            e, tokenizer, task_instruction, 512
        ),
        batched=True,
        remove_columns=train_data.column_names,
    )
    tokenized_val = val_data.map(
        lambda e: preprocess_samples(
            e, tokenizer, task_instruction, 512
        ),
        batched=True,
        remove_columns=val_data.column_names,
    )

    return tokenized_train, tokenized_val

In [28]:
tokenized_train, tokenized_val = tokenize_dataset(tokenizer, dataset["train"], dataset["validation"])


Tokenizing datasets...


Map:   0%|          | 0/146482 [00:00<?, ? examples/s]

Map:   0%|          | 0/41852 [00:00<?, ? examples/s]

In [30]:
!pip install wandb weave

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.9/853.9 kB 16.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 51.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.6/83.6 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 5.5.2
    Uninstalling cachetools-5.5.2:
      Successfully uninstalled cachetools-5.5.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-

In [31]:
import os
os.environ['WANDB_API_KEY'] = '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'

In [ ]:
collator = PaddingCollator(tokenizer=tokenizer)

output_dir = os.path.join(OUTPUTS_DIR, "lora_samsum")
os.makedirs(output_dir, exist_ok=True)

args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    max_steps=500,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=float(2e-4),
    lr_scheduler_type="cosine",
    warmup_steps=100,
    bf16=True,
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=25,
    save_total_limit=2,
    max_grad_norm=1.0,
    report_to="wandb",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
)

print("\nStarting LoRA fine-tuning...")
trainer.train()
print("\nTraining complete!")

save_dir = os.path.join(output_dir, "lora_adapters")
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Saved LoRA adapters to {save_dir}")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 


Starting LoRA fine-tuning...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: gr8nishan (gr8nishan-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept

Step,Training Loss,Validation Loss


In [ ]:
from huggingface_hub import login

login(token="<YOUR_HF_TOKEN>")

In [ ]:
def push_to_hub(
    model: PeftModel, tokenizer: AutoTokenizer, model_name: str, hf_username: str
):
    """
    Push a model and tokenizer to Hugging Face Hub.
    """
    model_id = f"{hf_username}/{model_name}"
    try:
        model.push_to_hub(f"{model_id}-adapters", private=False)

        merged_model = model.merge_and_unload()
        merged_model.push_to_hub(model_id, private=False)

        tokenizer.push_to_hub(model_id)
        print(f"Adapters successfully pushed to: https://huggingface.co/{model_id}")
    except Exception as e:
        print(f"Error pushing to Hugging Face: {e}")
        print("Make sure you're logged in with: huggingface-cli login")

In [ ]:
push_to_hub(model, tokenizer, "qwen-ner", "gr8nishan")